# 🛠️ Practical Assignment: Your First Complete ML Lifecycle
**Week 7 · Notebook 9 of 10 · ⏱️ 2–3 hours**

---

> 🎯 **Goal:** by the end of this notebook you will have personally carried a model through the *entire* lifecycle: from a raw CSV file to a clear, honest written explanation of how good it is. This is the skill the whole phase has been building toward.

## 📋 The scenario

You've just joined **Maplewood Realty**, a small startup. Your manager says:

> *"We want to give sellers an instant price estimate for their home. Build a model from our past sales data, tell me how accurate it is, and explain it to me in plain English; I'm not a data scientist."*

## 🪜 The seven steps

You will complete each step of the lifecycle with scikit-learn:

| # | Step | Lessons that prepared you |
|---|---|---|
| 1 | **Load** a structured dataset | 1 |
| 2 | **Separate** features and target | 1, 2 |
| 3 | **Split** the dataset | 4 |
| 4 | **Train** a regression model | 2, 4, 8 |
| 5 | **Generate** predictions | 2, 4 |
| 6 | **Evaluate** the model | 6 |
| 7 | **Explain** its performance | 5, 6 |

## 📏 How you'll be graded (100 points)

| Component | Points | What "full marks" looks like |
|---|---|---|
| Steps 1–3 (data handling) | 20 | correct shapes, correct split, **no leakage**, reproducible (`random_state`) |
| Steps 4–5 (model & predictions) | 20 | a fitted model *and* a baseline; predictions on both train and test |
| Step 6 (evaluation) | 25 | MAE, RMSE, R² on train **and** test, compared with the baseline, plus a residual plot |
| Step 7 (explanation) | 30 | plain-English, quantitative, honest about limits (see the writing guide in Step 7) |
| Code clarity | 5 | readable names, short comments, runs top-to-bottom with *Restart & Run All* |

*The self-check cells tell you if the mechanical parts are right (✅). The written explanation is graded by a human: or by you, using the checklist at the end.*

## 🧰 The dataset: `housing.csv`

900 past home sales. Each row is one house.

| Column | Meaning |
|---|---|
| `area_sqm` | living area in square metres |
| `bedrooms`, `bathrooms` | counts |
| `age_years` | age of the building |
| `distance_to_center_km` | distance to the city centre |
| `has_garden` | 1 = yes, 0 = no |
| `neighborhood_quality` | rating from 1 (poor) to 5 (excellent) |
| **`price_k`** | **sale price in $1,000s: our TARGET** |

> 💡 **Tip:** each step's starter cell uses *placeholders* (`None`). Replace them with your code. Later steps build on earlier ones, so work **in order**. If a self-check says ⏳, re-read the statement and open one hint.

In [1]:
import sys
sys.path.append("..")
from course_utils import *
setup()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### 🧩 Step 9.1: Load the dataset &nbsp; ⭐

**Load** `housing.csv` into a DataFrame called `df` (use `load("housing.csv")`). Then inspect it and record:

* `n_rows`: number of rows,
* `n_cols`: number of columns,
* `n_missing`: the total number of missing values in the whole table.

Also *look* at the data (`df.head()`, `df.describe()`, `df.dtypes`). Real projects begin with **looking**, not modelling. Does anything look odd: impossible values, weird ranges?

**Stuck?** Open the hints one at a time: try again between each one.

<details>
<summary>💡 Hint 1</summary>

`df.shape` gives `(rows, columns)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`df.isna().sum().sum()` counts every missing cell.

</details>

In [ ]:
# 1. load the data

df = None
n_rows = n_cols = n_missing = None

# 2. look at it (uncomment and run these)
# df.head()
# df.describe().T

In [ ]:
# ✔ Self-check - run this after you have solved the problem
check("df is the housing table", lambda: list(df.columns)[-1] == "price_k")
check("900 rows", lambda: n_rows == 900)
check("8 columns", lambda: n_cols == 8)
check("no missing values", lambda: n_missing == 0)

### 🧩 Step 9.2: Separate features and target &nbsp; ⭐

Create the feature table `X` (all columns **except** `price_k`) and the target `y` (the `price_k` column).

**Stuck?** Open the hints one at a time: try again between each one.

<details>
<summary>💡 Hint 1</summary>

`df.drop(columns="price_k")` and `df["price_k"]`.

</details>

In [ ]:
X = None
y = None

In [ ]:
# ✔ Self-check - run this after you have solved the problem
check("X has 7 feature columns", lambda: X.shape == (900, 7))
check("the target is NOT in X (no leakage!)", lambda: "price_k" not in X.columns)
check("y is the price column", lambda: y.name == "price_k" and len(y) == 900)

### 🧩 Step 9.3: Split the dataset &nbsp; ⭐⭐

Split into **training (80%)** and **test (20%)** sets using `train_test_split` with **`random_state=42`**. Name the pieces `X_train, X_test, y_train, y_test`.

*Reminder from Lesson 4: the test set is a sealed envelope. You will open it **once**, in Step 6.*

**Stuck?** Open the hints one at a time: try again between each one.

<details>
<summary>💡 Hint 1</summary>

`train_test_split(X, y, test_size=0.2, random_state=42)`; mind the order of the four outputs.

</details>

In [ ]:
X_train = X_test = y_train = y_test = None

In [ ]:
# ✔ Self-check - run this after you have solved the problem
check("720 training rows", lambda: len(X_train) == 720 and len(y_train) == 720)
check("180 test rows", lambda: len(X_test) == 180 and len(y_test) == 180)
check("same split as test_size=0.2, random_state=42", lambda: list(X_test.index) == list(train_test_split(X, y, test_size=0.2, random_state=42)[1].index))
check("no overlap between train and test", lambda: not set(X_train.index) & set(X_test.index))

### 🧩 Step 9.4: Train a regression model &nbsp; ⭐⭐

1. Create and **fit** a `LinearRegression` on the **training data only** → `model`.
2. Also fit a **baseline** `DummyRegressor(strategy="mean")` on the training data → `baseline`.
3. Build `coefs`: a pandas Series of the model's coefficients, indexed by feature name.

**Stuck?** Open the hints one at a time: try again between each one.

<details>
<summary>💡 Hint 1</summary>

`model = LinearRegression().fit(X_train, y_train)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`pd.Series(model.coef_, index=X_train.columns)`.

</details>

In [ ]:
model = None
baseline = None
coefs = None

In [ ]:
# ✔ Self-check - run this after you have solved the problem
check("model is a fitted LinearRegression with 7 coefficients", lambda: isinstance(model, LinearRegression) and len(model.coef_) == 7)
check("baseline is a fitted DummyRegressor", lambda: isinstance(baseline, DummyRegressor) and hasattr(baseline, "constant_"))
check("the model was trained on the training set only (compare intercepts)", lambda: abs(model.intercept_ - LinearRegression().fit(X_train, y_train).intercept_) < 1e-9)
check("coefs is a labelled Series; age lowers the price", lambda: isinstance(coefs, pd.Series) and coefs["age_years"] < 0)

### 🧩 Step 9.5: Generate predictions &nbsp; ⭐

Use the fitted `model` to predict prices for **both** the training and the test features:

* `y_pred_train`: predictions for `X_train`
* `y_pred_test`: predictions for `X_test`

Then also compute `y_base_test`, the **baseline's** predictions for `X_test`.

**Stuck?** Open the hints one at a time: try again between each one.

<details>
<summary>💡 Hint 1</summary>

`model.predict(X_test)`.

</details>

In [ ]:
y_pred_train = None
y_pred_test = None
y_base_test = None

In [ ]:
# ✔ Self-check - run this after you have solved the problem
check("one prediction per training row", lambda: len(y_pred_train) == 720)
check("one prediction per test row", lambda: len(y_pred_test) == 180)
check("test predictions come from the model", lambda: np.allclose(y_pred_test, model.predict(X_test)))
check("baseline predicts one constant value for everyone", lambda: np.allclose(y_base_test, y_base_test[0]) and len(y_base_test) == 180)

### 🧩 Step 9.6: Evaluate the model &nbsp; ⭐⭐⭐

Compute **MAE, RMSE and R²** for the model on both the **training** and the **test** sets, and store them in a dictionary:

```python
metrics = {"train": {"MAE": ..., "RMSE": ..., "R2": ...},
           "test":  {"MAE": ..., "RMSE": ..., "R2": ...}}
```

Also compute the **baseline's test RMSE** (`baseline_rmse`) and the **percentage improvement** of the model's test RMSE over it (`improvement_pct`, as in Lesson 6).

Finally, draw a **residual plot** for the test set (predicted on the x-axis, `actual − predicted` on the y-axis, with a horizontal line at 0). *A plot can't be auto-checked, but your Step 7 explanation will need it.*

**Stuck?** Open the hints one at a time: try again between each one.

<details>
<summary>💡 Hint 1</summary>

`mean_absolute_error`, `mean_squared_error` (take `np.sqrt` for RMSE) and `r2_score`, each called as `metric(y_true, y_pred)`.

</details>

<details>
<summary>💡 Hint 2</summary>

A helper function that takes `(y_true, y_pred)` and returns the three-metric dictionary saves repetition.

</details>

<details>
<summary>💡 Hint 3</summary>

`plt.scatter(y_pred_test, y_test - y_pred_test)` and `plt.axhline(0, color="red")`.

</details>

In [ ]:
def evaluate(y_true, y_pred):
    return {"MAE": None, "RMSE": None, "R2": None}

metrics = {"train": None, "test": None}
baseline_rmse = None
improvement_pct = None

# residual plot below

In [ ]:
# ✔ Self-check - run this after you have solved the problem
check("test RMSE is correct", lambda: abs(metrics["test"]["RMSE"] - np.sqrt(mean_squared_error(y_test, y_pred_test))) < 1e-9)
check("test MAE is correct", lambda: abs(metrics["test"]["MAE"] - mean_absolute_error(y_test, y_pred_test)) < 1e-9)
check("train R² is correct", lambda: abs(metrics["train"]["R2"] - r2_score(y_train, y_pred_train)) < 1e-9)
check("model is strong (test R² > 0.95)", lambda: metrics["test"]["R2"] > 0.95)
check("baseline RMSE is correct", lambda: abs(baseline_rmse - np.sqrt(mean_squared_error(y_test, y_base_test))) < 1e-9)
check("improvement over baseline is > 75%", lambda: improvement_pct > 75)

### 🧩 Step 9.7: Explain the performance &nbsp; ⭐⭐⭐

Your manager isn't a data scientist. Write a short, **plain-English** explanation by filling in each field of the dictionary `explanation` (one or two sentences each, **at least 60 characters per field**, and **quote numbers from your results**).

**Writing guide: good explanations are:**
* **Quantitative**: "typically off by about $26k (≈ 5%)", not "pretty accurate".
* **Comparative**: always against the baseline.
* **Honest**: mention limits and what you *didn't* test.
* **Jargon-free**: if you must say "RMSE", translate it.

Fields: `task`, `performance`, `baseline`, `generalization`, `errors`, `drivers`, `limitations`.

**Stuck?** Open the hints one at a time: try again between each one.

<details>
<summary>💡 Hint 1</summary>

`task`: what is being predicted and from what? `performance`: typical error in dollars and R² in words.

</details>

<details>
<summary>💡 Hint 2</summary>

`generalization`: compare train vs test numbers: is there a gap? `errors`: what does the residual plot show, and where does the model struggle?

</details>

<details>
<summary>💡 Hint 3</summary>

`drivers`: which features move the predicted price most, phrased as "holding all else equal…"; `limitations`: one data limitation (e.g. one city, one period) and one modelling limitation (e.g. linear, fan-shaped errors, no cross-validation).

</details>

In [ ]:
explanation = {
    "task": "",
    "performance": "",
    "baseline": "",
    "generalization": "",
    "errors": "",
    "drivers": "",
    "limitations": "",
}

In [ ]:
# ✔ Self-check - run this after you have solved the problem
fields = ["task", "performance", "baseline", "generalization", "errors", "drivers", "limitations"]
check("all seven fields exist", lambda: sorted(explanation) == sorted(fields))
for f in fields:
    check(f"'{f}' has at least 60 characters", lambda: len(explanation[f].strip()) >= 60)
check("you quoted at least five numbers in total", lambda: sum(ch.isdigit() for ch in " ".join(explanation.values())) >= 10)

---
## ✅ Self-assessment checklist

Before you call this done, tick every box:

- [ ] The notebook runs top-to-bottom after **Kernel → Restart & Run All**.
- [ ] The target never appears in `X`; the model was fitted on **training data only**.
- [ ] I set `random_state` wherever randomness appears.
- [ ] I reported **MAE, RMSE and R²** on **both** train and test and compared them to the **baseline**.
- [ ] I looked at a **residual plot** and said something meaningful about it.
- [ ] My explanation is understandable to a non-technical reader and includes numbers and limitations.

## 🚀 Stretch challenges (optional, ungraded)

1. **Cross-validate.** Replace the single split by 5-fold cross-validation (`cross_val_score`) and report mean ± std of RMSE.
2. **Regularise.** Try `Ridge` / `Lasso` (inside `make_pipeline(StandardScaler(), …)`). Does anything improve? Why or why not?
3. **Transform the target.** The residual plot fans out: try fitting the *log* of the price. (Hint: `np.log`, then `np.exp` on predictions.) Does the fan shrink?
4. **Feature engineering.** Add `area_sqm * neighborhood_quality` as a new feature. Does the test RMSE drop?
5. **Gradient descent.** Re-implement the fit with your Lesson 8 `gradient_descent` on standardised features and confirm it matches scikit-learn.
6. **Different dataset.** Repeat the whole lifecycle on `diabetes.csv` (target `disease_progression`). How does R² compare, and what does that tell you about the problem?

## 🎉 You did it

You have now run the complete lifecycle: **load → separate → split → train → predict → evaluate → explain.** Every machine-learning system in the world (from a spam filter to a self-driving car) is a bigger, fancier version of exactly what you just did.

**Next up →** `10_capstone_project.ipynb`: messier data, missing values, categories, model comparison and a full written report. Time to show what you've got.